# CAP F1 Abstention
This notebook (hackily) adapts Jimin's CAP F1 code to use with our self-consistency based absention work.

## Include Library

In [1]:
from datetime import datetime
import os
import random

# library for cap_f1
from cap_f1 import LLMClient, AtomicProcessor, ResultsRepo
from fewshot_examples import (
    FEWSHOT_DEDUP_MESSAGES,
    FEWSHOT_RECALL_MESSAGES,
    FEWSHOT_PRECISION_MESSAGES,
)

# code for no need for restarting the kernel when python file is updated
%load_ext autoreload
%autoreload 2

## 1. build API + processor (inject few-shot examples if you want)
currently fewshot example is at fewshot_examples.py

In [2]:
# 1) build API + processor (inject few-shot examples if you want)
llm = LLMClient()
proc = AtomicProcessor(
    llm,
    fewshot_dedup=FEWSHOT_DEDUP_MESSAGES,  # or None
    fewshot_recall=FEWSHOT_RECALL_MESSAGES,  # or None
    fewshot_precision=FEWSHOT_PRECISION_MESSAGES,  # or None
)

Initialized LLM client with gpt-4.1-2025-04-14 and temperature = 0.2


## 2. Load Data

In [3]:
def get_random_sample(dataset, count=1):
    """Get a random sample from the dataset

    Args:
        dataset (list): List of dictionaries
        count (int, optional): Number of samples to return. Defaults to 1.

    Returns:
        list: List of dictionaries (random samples)
    """
    return random.sample(dataset, count)

In [4]:
print("Loading abstention dataset...")

# for filename
now = datetime.now()
timestamp = now.strftime("%Y-%m-%d_%H-%M")

# create folder to save the results
folder_path = f"results/{timestamp}"
os.makedirs(folder_path, exist_ok=True)

# features that we need to extract from the original dataset
org_caption_dataset = ResultsRepo.read_json(
    "test-data-scored-with-mt-metrics_300-examples_2025-08-12_12-29-55.json"
)

# number of data points testing
LIMIT = len(org_caption_dataset)
org_caption_dataset = org_caption_dataset[:LIMIT]
# org_caption_dataset = get_random_sample(org_caption_dataset, count=LIMIT)

# show dataset
print(f"Number of data points: {len(org_caption_dataset)}")
org_caption_dataset[0]
# org_caption_dataset

Loading abstention dataset...
Number of data points: 300


{'image_id': 1308,
 'file_name': 'VizWiz_train_00001308.jpg',
 'vizwiz_url': 'https://vizwiz.cs.colorado.edu/VizWiz_visualization_img/VizWiz_train_00001308.jpg',
 'human_captions': 'a small 330 ml bottle of praise French salad dressing\nA picture of the food is on the packaging.\nSmall blue and green French dressing bottle laying sideways.\nA person holding a bottle of French dressing by Praise in green and blue lettering on a tan placemat.\na bottle of French salad dressing brand by praise',
 'annotator': 'Anne Marie',
 'annotation': 'praise french dressing',
 'gpt4o_caption': 'A bottle of Praise French dressing with a green cap, featuring a blue label with text "French" and product details. The bottle is being held over a woven mat.',
 'gpt4o_code': 'yes',
 'greedy_response': 'A bottle of Praise French dressing with a green cap and a blue label. The label includes images of garlic and herbs, and text indicating it is a 330ml bottle with no preservatives, artificial colors, or flavors

### Hacks for getting abstention data to work with current CAP F1

1. Change human_captions to human_captions_crowdworkers
2. Create a new human_captions field with the following:

```python
'human_captions': [
    {
        'caption': <greedy caption goes here>,
        'is_precanned': False,
        'is_rejected': False
    }
]
```
3. Create a model_captions to hold samples as:
```python
'model_captions': [
    {
        'model_name': 'sample_1',
        'caption': ''
    },
    {
        'model_name': 'sample_2',
        'caption': ''
    },
    ...
    {
        'model_name': 'sample_10',
        'caption': ''
    },
],
```

In [5]:
for item in org_caption_dataset:
    # hack 1
    item["crowdworker_captions"] = item["human_captions"]
    del item["human_captions"]

    # hack 2
    item["human_captions"] = [
        {
            "caption": item["greedy_response"],
            "is_precanned": False,
            "is_rejected": False,
        }
    ]

    # hack 3
    item["model_captions"] = [
        {
            "model_name": f"sample_{index + 1}",
            "caption": sample,
        }
        for index, sample in enumerate(item["additional_responses"])
    ]

    # add an evaluation field
    item["evaluation"] = {}
org_caption_dataset[0]

{'image_id': 1308,
 'file_name': 'VizWiz_train_00001308.jpg',
 'vizwiz_url': 'https://vizwiz.cs.colorado.edu/VizWiz_visualization_img/VizWiz_train_00001308.jpg',
 'annotator': 'Anne Marie',
 'annotation': 'praise french dressing',
 'gpt4o_caption': 'A bottle of Praise French dressing with a green cap, featuring a blue label with text "French" and product details. The bottle is being held over a woven mat.',
 'gpt4o_code': 'yes',
 'greedy_response': 'A bottle of Praise French dressing with a green cap and a blue label. The label includes images of garlic and herbs, and text indicating it is a 330ml bottle with no preservatives, artificial colors, or flavors.',
 'additional_responses': ['A bottle of Praise brand French dressing with a green cap, held in a hand. The label is blue with images of garlic and herbs, and text highlighting "No Preservatives," "No Artificial Colours," and "No Artificial Flavours."',
  'A bottle of Praise French dressing with a green cap and a blue label displayi

### 3. generate atomics

In [ ]:
# 3) generate atomics
print(f"Generating atomic statements using {llm.model}")
T_atomics, g_atomics, parsed_T = proc.generate_atomic_statement(
    org_caption_dataset, limit=LIMIT
)

# hack: if a T_atomic is None, set it to parsed_T
for index, curr_t_atomic in enumerate(T_atomics):
    if curr_t_atomic is None:
        T_atomics[index] = {"atomic_captions": parsed_T[index]}

# 3.1) save intermediate
print("Saving intermediate results...")
all_human_captions = []
for item in org_caption_dataset:
    # Filter out human captions that are mention quality issues
    human_captions = [
        hc["caption"]
        for hc in item["human_captions"]
        if hc["caption"] != "Quality issues are too severe to recognize visual content."
    ]
    all_human_captions.append(human_captions)
ResultsRepo.save_results_json(
    output_path=f"{folder_path}/intermediate_{timestamp}.json",
    org_dataset=org_caption_dataset,
    T_atomics=T_atomics,
    g_atomics=g_atomics,
    parsed_T=parsed_T,
    T_org=all_human_captions,
    limit=LIMIT,
)

Generating atomic statements using gpt-4.1-2025-04-14


### 4. evaluate and get recall and precision
- match human caption to model caption
- create recall and precision data

In [33]:
# before calculating F1 score, match sentences between human generated and model generated
print("Evaluating atomic statements...")
eval_out = proc.evaluate_matching(all_human_captions, T_atomics, g_atomics)

# 4.1) save evaluation results
ResultsRepo.save_results_json(
    output_path=f"{folder_path}/eval_{timestamp}.json",
    update_existing=f"{folder_path}/intermediate_{timestamp}.json",
    metadata=eval_out,
    limit=LIMIT,
)

Evaluating atomic statements...


  1%|          | 3/300 [03:28<5:12:44, 63.18s/it] 

[sample_1] Recall mismatch: len T=5 vs TP+FN=5
[sample_3] Recall mismatch: len T=5 vs TP+FN=5
[sample_5] Recall mismatch: len T=5 vs TP+FN=5
[sample_7] Recall mismatch: len T=5 vs TP+FN=5
[sample_9] Recall mismatch: len T=5 vs TP+FN=5


  1%|▏         | 4/300 [04:15<4:41:34, 57.08s/it]

[sample_10] Recall mismatch: len T=5 vs TP+FN=5
[sample_1] Recall mismatch: len T=6 vs TP+FN=6
[sample_7] Recall mismatch: len T=6 vs TP+FN=6


...

[sample_7] Precision mismatch: len G=11 vs TP+FP=11


 97%|█████████▋| 290/300 [4:58:15<09:46, 58.69s/it]

[sample_2] Recall mismatch: len T=7 vs TP+FN=8


100%|██████████| 300/300 [5:07:37<00:00, 61.53s/it]


Saved JSON to: results/2025-08-25_00-03/eval_2025-08-25_00-03.json


### 5. calculate cap f1 score


In [34]:
# 5) calculate cap f1 score
cap_scores = proc.calculate_cap_f1(eval_out)

# 5.1) save cap f1 score results
ResultsRepo.save_results_json(
    output_path=f"{folder_path}/final_{timestamp}.json",
    update_existing=f"{folder_path}/eval_{timestamp}.json",
    evaluations=cap_scores,
    limit=LIMIT,
)

100%|██████████| 300/300 [00:00<00:00, 138960.93it/s]


Saved JSON to: results/2025-08-25_00-03/final_2025-08-25_00-03.json


In [35]:
# 6) Final JSON → CSV
print("Saving final results into csv...")
ResultsRepo.export_final_csv(
    json_path=f"{folder_path}/final_{timestamp}.json",
    csv_path=f"{folder_path}/final_{timestamp}.csv",
    # model_keys={"gpt":"gpt-4o-2024-08-06", "molmo":"Molmo-7B-O-0924", "llama":"Llama-3.2-11B-Vision-Instruct"}
)

Saving final results into csv...
CSV file saved to: results/2025-08-25_00-03/final_2025-08-25_00-03.csv


### Export as formatted CSV

In [1]:
# import json
import json
import pandas as pd

output_data = json.load(open(f"{folder_path}/eval_{timestamp}.json"))

min_f1 = 0.0
max_f1 = 0.0
for output in output_data:
    keys_to_remove = [
        "expected_match_count_scores",
        "match_count_scores",
        "greedy_response_no_stop",
        "additional_responses_no_stop",
        "bleu-1",
        "bleu-2",
        "bleu-3",
        "bleu-4",
        "meteor",
        "rouge",
        "cider",
        "spice",
        "bertscore",
        "bertscore_idf",
        "crowdworker_captions",
        "human_captions",
        "model_captions",
    ]

    for key in keys_to_remove:
        del output[key]

    # expand additionaal_responses into separate keys as sample_1 .. sample_n
    additional_responses = output["additional_responses"]
    for i, response in enumerate(additional_responses):
        output[f"sample_{i + 1}"] = response
    del output["additional_responses"]

    # get t_atomics
    output["t_atomics"] = output["evaluation"]["cap_f1"]["T_atomics"]

    # get g_atomics for each sample
    all_g_atomics = output["evaluation"]["cap_f1"]["g_atomics"]
    for key in all_g_atomics.keys():
        output[f"{key}_g_atomics"] = all_g_atomics[key]

    # get all recall and precisions scores
    all_scores = output["evaluation"]["cap_f1"]["scores"]
    all_recall = []
    all_precision = []
    for key in all_scores.keys():
        output[f"{key}_recall"] = all_scores[key]["recall"]
        output[f"{key}_precision"] = all_scores[key]["precision"]

        all_recall.append(all_scores[key]["recall"])
        all_precision.append(all_scores[key]["precision"])

    # compute average recall and precision
    output["average_recall"] = sum(all_recall) / len(all_recall)
    output["average_precision"] = sum(all_precision) / len(all_precision)

    # compute average f1 as the harmonic mean of recall and precision
    output["average_f1"] = (
        2
        * (output["average_recall"] * output["average_precision"])
        / (output["average_recall"] + output["average_precision"])
    )

    min_f1 = min(min_f1, output["average_f1"])
    max_f1 = max(max_f1, output["average_f1"])

# normalize f1 score
all_f1 = []
for output in output_data:
    output["normalized_f1"] = (output["average_f1"] - min_f1) / (max_f1 - min_f1)

# save as csv
pd.DataFrame(output_data).to_csv(
    "./results/2025-08-20_23-23/final_2025-08-20_23-23_cleaned.csv", index=False
)

NameError: name 'folder_path' is not defined